# EDA — Exploratory Data Analysis

Analysis of sample videos: resolution, FPS, duration, object class distribution, event statistics.

In [ ]:
import cv2
import json
from pathlib import Path
from collections import Counter

videos = ['../traffic_short.mp4', '../traffic_short02.mp4']
for v in videos:
    if not Path(v).exists():
        print(f'{v}: not found')
        continue
    cap = cv2.VideoCapture(v)
    fps = cap.get(cv2.CAP_PROP_FPS)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    print(f'{v}: {w}x{h}, {fps:.1f} FPS, {n} frames, {n/fps:.1f}s')
    cap.release()

## Object detection on sample frame

In [ ]:
from ultralytics import YOLO

model = YOLO('../weights/model.pt')
for v in videos:
    if not Path(v).exists():
        continue
    cap = cv2.VideoCapture(v)
    cap.set(cv2.CAP_PROP_POS_FRAMES, 30)
    ret, frame = cap.read()
    cap.release()
    if ret:
        res = model.predict(frame, conf=0.25, verbose=False)[0]
        classes = [model.names[int(c)] for c in res.boxes.cls]
        print(f'{v}: {dict(Counter(classes))}')

## Detected events (from predictions_samples.json)

In [ ]:
with open('../predictions_samples.json') as f:
    samples = json.load(f)

for s in samples['samples']:
    print(f"\n{s['description']}")
    print(f"  Total events: {s['events_count']}")
    types = Counter(e['event_type'] for e in s['predictions']['events'])
    for t, n in types.most_common():
        print(f'    {t}: {n}')